In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

MODEL_FOLDER = '/content/drive/My Drive/Food101_Models'
os.makedirs(MODEL_FOLDER, exist_ok=True)

Mounted at /content/drive


In [ ]:
!pip install kagglehub
!pip install requests
!pip install tensorflow

import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import kagglehub
import requests
import time
from typing import Dict, Optional, List

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")
print("All dependencies installed and imported successfully!")

TensorFlow version: 2.20.0
GPU Available: []
All dependencies installed and imported successfully!


In [ ]:
path = kagglehub.dataset_download("dansbecker/food-101")

print("Path to dataset files:", path)

Resuming download from 620756992 bytes (9446302140 bytes left)...
Resuming download to /root/.cache/kagglehub/datasets/dansbecker/food-101/1.archive (620756992/10067059132) bytes left.


100%|██████████| 9.38G/9.38G [02:23<00:00, 65.7MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/dansbecker/food-101/versions/1


In [ ]:
dataset_root = os.path.join(path, "food-101", "food-101")

print(f"Contents of {dataset_root}: {os.listdir(dataset_root)}")


images_dir = os.path.join(dataset_root, "images")
meta_dir = os.path.join(dataset_root, "meta")

print(f"\nDataset root: {dataset_root}")
print(f"Images directory exists: {os.path.exists(images_dir)}")
print(f"Meta directory exists: {os.path.exists(meta_dir)}")

food_categories = sorted([d for d in os.listdir(images_dir) if os.path.isdir(os.path.join(images_dir, d))])
num_categories = len(food_categories)
print(f"\nNumber of food categories: {num_categories}")
print(f"First 10 categories: {food_categories[:10]}")
print(f"Last 10 categories: {food_categories[-10:]}")

Contents of /root/.cache/kagglehub/datasets/dansbecker/food-101/versions/1/food-101/food-101: ['images', 'license_agreement.txt', 'meta', '.DS_Store', 'README.txt']

Dataset root: /root/.cache/kagglehub/datasets/dansbecker/food-101/versions/1/food-101/food-101
Images directory exists: True
Meta directory exists: True

Number of food categories: 101
First 10 categories: ['apple_pie', 'baby_back_ribs', 'baklava', 'beef_carpaccio', 'beef_tartare', 'beet_salad', 'beignets', 'bibimbap', 'bread_pudding', 'breakfast_burrito']
Last 10 categories: ['spaghetti_carbonara', 'spring_rolls', 'steak', 'strawberry_shortcake', 'sushi', 'tacos', 'takoyaki', 'tiramisu', 'tuna_tartare', 'waffles']


In [ ]:


print("Loading trained model, class names, and calorie database...")

class_names_path = os.path.join(MODEL_FOLDER, "class_names.json")
with open(class_names_path, "r") as f:
    class_names = json.load(f)

calorie_db_path = os.path.join(MODEL_FOLDER, "calorie_database.json")
with open(calorie_db_path, "r") as f:
    calorie_database = json.load(f)

model_path = os.path.join(MODEL_FOLDER, 'food_recognition_efficientnet_fine_tuned.h5')


print(f"✓ Loaded {len(class_names)} food classes.")
print(f"✓ Loaded calorie data for {len(calorie_database)} foods.")


Loading trained model, class names, and calorie database...
✓ Loaded 101 food classes.
✓ Loaded calorie data for 101 foods.


### Load the previously saved model

In [ ]:


from google.colab import drive

drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

# Verify the folder exists
import os
MODEL_FOLDER = '/content/drive/My Drive/Food101_Models'

if os.path.exists(MODEL_FOLDER):
    print(f"✅ Found folder: {MODEL_FOLDER}")
    print(f"✅ Files in folder:")
    for file in os.listdir(MODEL_FOLDER):
        print(f"   - {file}")
else:
    print(f"❌ Folder not found: {MODEL_FOLDER}")
    print("   Check that the exact folder name matches your Google Drive")

# ============================================================================
# NOW LOAD THE MODEL
# ============================================================================

model_path = os.path.join(MODEL_FOLDER, 'food_recognition_efficientnetb0_fine_tuned_interrupted.h5')

try:
    model = tf.keras.models.load_model(model_path)
    print(f"✅ Model loaded successfully from: {model_path}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("Please ensure the model file exists at the specified path in your Google Drive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully!
✅ Found folder: /content/drive/My Drive/Food101_Models
✅ Files in folder:
   - usda_database_builder.py
   - food_recognition_efficientnetb0_fine_tuned_interrupted.h5
   - history_phase1.json
   - history_phase2.json
   - food_recognition_model.h5
   - class_names.json
   - calorie_database.json
   - food_recognition_efficientnetb0_fine_tuned.h5
   - food_classifier_app.py


✅ Model loaded successfully from: /content/drive/My Drive/Food101_Models/food_recognition_efficientnetb0_fine_tuned_interrupted.h5


In [ ]:

model_path = os.path.join(MODEL_FOLDER, 'food_recognition_efficientnetb0_fine_tuned_interrupted.h5')

try:
    model = tf.keras.models.load_model(model_path)
    print(f"✓ Model loaded successfully from: {model_path}")
except Exception as e:
    print(f"❌ Error loading model: {e}")
    print("Please ensure the model file exists at the specified path in your Google Drive.")

✓ Model loaded successfully from: /content/drive/My Drive/Food101_Models/food_recognition_efficientnetb0_fine_tuned_interrupted.h5


In [ ]:
print(f"Contents of MODEL_FOLDER ({MODEL_FOLDER}):")
if os.path.exists(MODEL_FOLDER):
    for item in os.listdir(MODEL_FOLDER):
        print(f"- {item}")
else:
    print(f"The directory {MODEL_FOLDER} does not exist.")

Contents of MODEL_FOLDER (/content/drive/My Drive/Food101_Models):
- usda_database_builder.py
- food_recognition_efficientnetb0_fine_tuned_interrupted.h5
- history_phase1.json
- history_phase2.json
- food_recognition_model.h5
- class_names.json
- calorie_database.json
- food_recognition_efficientnetb0_fine_tuned.h5
- food_classifier_app.py


### Re-running Model Training and Saving

It seems the model was not successfully saved in the previous run. This section will re-execute the training cell to ensure the model is trained and saved correctly to Google Drive.

In [ ]:


print("Creating data pipeline with image loading and augmentation...")

import random
from tensorflow.keras.applications.efficientnet import preprocess_input
IMG_HEIGHT = 160
IMG_WIDTH = 160
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = len(food_categories)


label_to_idx = {name: i for i, name in enumerate(food_categories)}


def load_paths_and_labels(filepath, base_images_dir, label_map):
    images_list = []
    labels_list = []
    with open(filepath, 'r') as f:
        for line in f:
            image_id_full = line.strip()
            parts = image_id_full.split('/')
            class_name = parts[0]
            image_id = parts[1]
            img_filename = f"{image_id}.jpg"
            full_image_path = os.path.join(base_images_dir, class_name, img_filename)
            images_list.append(full_image_path)
            labels_list.append(label_map[class_name])
    return images_list, labels_list


train_meta_path = os.path.join(meta_dir, "train.txt")
train_images_raw, train_labels_idx_raw = load_paths_and_labels(train_meta_path, images_dir, label_to_idx)

val_meta_path = os.path.join(meta_dir, "test.txt")
val_images_raw, val_labels_idx_raw = load_paths_and_labels(val_meta_path, images_dir, label_to_idx)


train_images, train_labels_idx = [], []
for img_path, label in zip(train_images_raw, train_labels_idx_raw):
    if os.path.exists(img_path):
        train_images.append(img_path)
        train_labels_idx.append(label)

val_images, val_labels_idx = [], []
for img_path, label in zip(val_images_raw, val_labels_idx_raw):
    if os.path.exists(img_path):
        val_images.append(img_path)
        val_labels_idx.append(label)

print(f"Total training images: {len(train_images)}")
print(f"Total validation images: {len(val_images)}")


print("\n=== SHUFFLING AND VERIFYING LABELS ===")
random.seed(SEED)

combined_train = list(zip(train_images, train_labels_idx))
random.shuffle(combined_train)
train_images, train_labels_idx = zip(*combined_train)
train_images, train_labels_idx = list(train_images), list(train_labels_idx)

combined_val = list(zip(val_images, val_labels_idx))
random.shuffle(combined_val)
val_images, val_labels_idx = zip(*combined_val)
val_images, val_labels_idx = list(val_images), list(val_labels_idx)


print(f"First 10 training labels: {train_labels_idx[:10]}")
print(f"Unique labels in first 100: {len(set(train_labels_idx[:100]))}")
print(f"Max label: {max(train_labels_idx)}, Min label: {min(train_labels_idx)}")
print(f"Unique labels total: {len(set(train_labels_idx))}/{NUM_CLASSES}")

def load_image(path, label):
    """Load and preprocess image for EfficientNetB0"""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = preprocess_input(img)
    return img, label

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
])

print("\nBuilding training dataset...")
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels_idx))
train_dataset = train_dataset.shuffle(buffer_size=1000, seed=SEED)
train_dataset = train_dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
)
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
print("Building validation dataset...")
val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_labels_idx))
val_dataset = val_dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE)
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

print("✓ Data pipelines created successfully!")


Creating data pipeline with image loading and augmentation...
Total training images: 75750
Total validation images: 25250

=== SHUFFLING AND VERIFYING LABELS ===
First 10 training labels: [18, 25, 33, 68, 50, 30, 62, 79, 93, 46]
Unique labels in first 100: 64
Max label: 100, Min label: 0
Unique labels total: 101/101

Building training dataset...
Building validation dataset...
✓ Data pipelines created successfully!


In [ ]:
import collections


train_category_counts = collections.Counter(train_labels_idx)


val_category_counts = collections.Counter(val_labels_idx)

print("\n--- Images per Category (Training Set) ---")

for i, (label_idx, count) in enumerate(sorted(train_category_counts.items(), key=lambda item: item[1], reverse=True)):
    if i < 5 or i >= len(food_categories) - 5:
        print(f"{food_categories[label_idx].replace('_', ' ').title()}: {count} images")
    elif i == 5:
        print("... (truncated) ...")

print("\n--- Images per Category (Validation Set) ---")
for i, (label_idx, count) in enumerate(sorted(val_category_counts.items(), key=lambda item: item[1], reverse=True)):
    if i < 5 or i >= len(food_categories) - 5:
        print(f"{food_categories[label_idx].replace('_', ' ').title()}: {count} images")
    elif i == 5:
        print("... (truncated) ...")


all_train_equal = len(set(train_category_counts.values())) == 1
all_val_equal = len(set(val_category_counts.values())) == 1

if all_train_equal:
    print(f"\nEach training category has {list(train_category_counts.values())[0]} images.")
else:
    print("\nTraining categories have varying image counts.")

if all_val_equal:
    print(f"Each validation category has {list(val_category_counts.values())[0]} images.")
else:
    print("\nValidation categories have varying image counts.")

print("\nTotal unique categories in training set:", len(train_category_counts))
print("Total unique categories in validation set:", len(val_category_counts))


--- Images per Category (Training Set) ---
Chicken Curry: 750 images
Club Sandwich: 750 images
Edamame: 750 images
Onion Rings: 750 images
Grilled Salmon: 750 images
... (truncated) ...
Cup Cakes: 750 images
Spaghetti Bolognese: 750 images
Guacamole: 750 images
Fish And Chips: 750 images
Croque Madame: 750 images

--- Images per Category (Validation Set) ---
Edamame: 250 images
Macarons: 250 images
Shrimp And Grits: 250 images
Pho: 250 images
Caesar Salad: 250 images
... (truncated) ...
Peking Duck: 250 images
French Fries: 250 images
Risotto: 250 images
Hot And Sour Soup: 250 images
Creme Brulee: 250 images

Each training category has 750 images.
Each validation category has 250 images.

Total unique categories in training set: 101
Total unique categories in validation set: 101


In [ ]:

#Diagnostic
print("\n--- Verifying Image File Paths ---")

example_missing_path = "/root/.cache/kagglehub/datasets/dansbecker/food-101/versions/1/food-101/food-101/images/apple_pie/apple_pie_1005649.jpg"
example_category_dir = os.path.dirname(example_missing_path)

print(f"Checking existence of example missing file: {example_missing_path}")
print(f"Does it exist? {os.path.exists(example_missing_path)}")

print(f"\nListing contents of parent directory: {example_category_dir}")
if os.path.exists(example_category_dir):
    print(os.listdir(example_category_dir)[:10])
    print(f"Total files in {os.path.basename(example_category_dir)}: {len(os.listdir(example_category_dir))}")
else:
    print(f"Directory does not exist: {example_category_dir}")

print(f"\nListing first 10 items in top-level images directory: {images_dir}")
if os.path.exists(images_dir):
    print(os.listdir(images_dir)[:10])
else:
    print(f"Top-level images directory does not exist: {images_dir}")


In [ ]:
#Diagnose: What does data look like
for images, labels in train_dataset.take(1):
    print("=== TRAIN BATCH ===")
    print(f"Image shape  : {images.shape}")
    print(f"Label shape  : {labels.shape}")
    print(f"Label dtype  : {labels.dtype}")
    print(f"Sample label : {labels[0].numpy()}")
    print(f"Label sum    : {labels[0].numpy().sum()}")  # Should be 1.0 if one-hot
    print(f"Image min/max: {images.numpy().min():.2f} / {images.numpy().max():.2f}")

# Check a batch from val
for images, labels in val_dataset.take(1):
    print("\n=== VAL BATCH ===")
    print(f"Image shape  : {images.shape}")
    print(f"Label shape  : {labels.shape}")
    print(f"Label dtype  : {labels.dtype}")
    print(f"Sample label : {labels[0].numpy()}")
    print(f"Label sum    : {labels[0].numpy().sum()}")  # Should be 1.0 if one-hot
    print(f"Image min/max: {images.numpy().min():.2f} / {images.numpy().max():.2f}")

In [ ]:
# After loading, add:
print("\n=== LABEL MAPPING DEBUG ===")
print(f"label_to_idx keys (first 10): {list(label_to_idx.keys())[:10]}")
print(f"Total categories in map: {len(label_to_idx)}")

# Check what's being parsed from train.txt
print("\n=== PARSING DEBUG ===")
with open(train_meta_path, 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(f"Line {i}: '{line.strip()}'")
            parts = line.strip().split('/')
            print(f"  Parsed parts: {parts}")
            print(f"  Class name: '{parts[0]}'")
            print(f"  Label index: {label_to_idx.get(parts[0], 'NOT FOUND')}")


In [ ]:
print("\n=== DEBUG INFO ===")
print(f"NUM_CLASSES: {NUM_CLASSES}")
print(f"Food categories count: {len(food_categories)}")

# Check a sample batch
for images, labels in train_dataset.take(1):
    print(f"Images shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Sample labels: {labels[:5]}")
    print(f"Label range: {tf.reduce_min(labels).numpy()}-{tf.reduce_max(labels).numpy()}")
    print(f"Image min/max: {tf.reduce_min(images).numpy():.2f}/{tf.reduce_max(images).numpy():.2f}")


In [ ]:
# Check GPU
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("GPU Details:", tf.sysconfig.get_build_info()['cuda_version'])

# Check dataset speed
print("\n⏱️  Testing dataset speed...")
import time

start = time.time()
for batch in train_dataset.take(1):
    print(f"Batch shape: {batch[0].shape}")
elapsed = time.time() - start
print(f"First batch took: {elapsed:.2f} seconds")

# Estimate total time
num_batches = len(train_images) // BATCH_SIZE
estimated_epoch_time = elapsed * num_batches
print(f"Estimated time per epoch: {estimated_epoch_time:.0f} seconds ({estimated_epoch_time/60:.1f} minutes)")


In [ ]:


import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import json
import time

print("="*80)
print("BUILDING EFFICIENTNETB0 MODEL - PRODUCTION VERSION")
print("="*80)

IMG_HEIGHT = 160
IMG_WIDTH = 160
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = len(food_categories)

# Load pre-trained EfficientNetB0
print("\n📥 Loading EfficientNetB0 from ImageNet...")
base_model = EfficientNetB0(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights='imagenet'
)

print("\n" + "="*80)
print("PHASE 1: FEATURE EXTRACTION (BASE MODEL FROZEN)")
print("="*80)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Model Architecture ---")
model.summary()

early_stopping_phase1 = EarlyStopping(
    monitor='val_accuracy',
    node=max,
    patience=12,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0005
)

reduce_lr_phase1 = ReduceLROnPlateau(
    monitor='val_accuracy',
    mode='max',
    factor=0.8,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

print("\n--- Starting Phase 1 Training ---")
print(f"⏱️  Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}")

start_time_p1 = time.time()
history_phase1 = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=[early_stopping_phase1, reduce_lr_phase1],
    verbose=1
)
phase1_time = time.time() - start_time_p1

print(f"\n✅ PHASE 1 COMPLETE! ({phase1_time/60:.1f} minutes)")
print(f"   Training Accuracy:   {history_phase1.history['accuracy'][-1]:.4f}")
print(f"   Validation Accuracy: {history_phase1.history['val_accuracy'][-1]:.4f}")
print(f"   Best Val Accuracy:   {max(history_phase1.history['val_accuracy']):.4f}")

print("\n" + "="*80)
print("PHASE 2: FINE-TUNING (UNFREEZING LAST 30 LAYERS)")
print("="*80)

# Unfreeze last 30 layers ONLY
num_layers = len(base_model.layers)
unfreeze_from = max(0, num_layers - 30)

print(f"\nTotal base model layers: {num_layers}")
print(f"Unfreezing from layer {unfreeze_from} onwards...")

for i, layer in enumerate(base_model.layers):
    if i >= unfreeze_from:
        layer.trainable = True
    # Keep batch norm frozen
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

# Count trainable params
trainable_count = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"✅ Trainable parameters in Phase 2: {trainable_count:,}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),  # ⬅️ Even LOWER LR
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n--- Model Summary (Phase 2) ---")
model.summary()


early_stopping_phase2 = EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=15,
    restore_best_weights=True,
    verbose=1,
    min_delta=0.0003
)

reduce_lr_phase2 = ReduceLROnPlateau(
    monitor='val_accuracy',
    mode='max',
    factor=0.85,
    patience=4,
    min_lr=1e-8,
    verbose=1
)

print("\n--- Starting Phase 2 Training ---")
print(f"⏱️  Timestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}")

start_time_p2 = time.time()
history_phase2 = model.fit(
    train_dataset,
    epochs=10,
    validation_data=val_dataset,
    callbacks=[early_stopping_phase2, reduce_lr_phase2],
    verbose=1
)
phase2_time = time.time() - start_time_p2

print(f"\n✅ PHASE 2 COMPLETE! ({phase2_time/60:.1f} minutes)")
print(f"   Training Accuracy:   {history_phase2.history['accuracy'][-1]:.4f}")
print(f"   Validation Accuracy: {history_phase2.history['val_accuracy'][-1]:.4f}")
print(f"   Best Val Accuracy:   {max(history_phase2.history['val_accuracy']):.4f}")

total_time = phase1_time + phase2_time
print(f"\n⏱️  Total Training Time: {total_time/3600:.2f} hours ({total_time/60:.1f} minutes)")

print("\n" + "="*80)
print("✅ TRAINING COMPLETE - READY FOR PRESENTATION!")
print("="*80)

# --- Save Model ---
model_path = os.path.join(MODEL_FOLDER, 'food_recognition_efficientnetb0_fine_tuned.h5')
model.save(model_path)
print(f"\n✅ Model saved: {model_path}")

# --- Save Histories ---
history_phase1_dict = {
    key: [float(val) for val in history_phase1.history[key]]
    for key in history_phase1.history
}
history_phase2_dict = {
    key: [float(val) for val in history_phase2.history[key]]
    for key in history_phase2.history
}

history_phase1_path = os.path.join(MODEL_FOLDER, 'history_phase1.json')
history_phase2_path = os.path.join(MODEL_FOLDER, 'history_phase2.json')

with open(history_phase1_path, 'w') as f:
    json.dump(history_phase1_dict, f, indent=2)

with open(history_phase2_path, 'w') as f:
    json.dump(history_phase2_dict, f, indent=2)

print(f"✅ Histories saved:")
print(f"   - {history_phase1_path}")
print(f"   - {history_phase2_path}")

# --- Print Summary for Presentation ---
print("\n" + "="*80)
print("📊 TRAINING SUMMARY FOR PRESENTATION")
print("="*80)
print(f"\nPhase 1 (Feature Extraction):")
print(f"  - Epochs run: {len(history_phase1.history['accuracy'])}")
print(f"  - Time: {phase1_time/60:.1f} minutes")
print(f"  - Final Val Acc: {history_phase1.history['val_accuracy'][-1]:.2%}")
print(f"\nPhase 2 (Fine-tuning):")
print(f"  - Epochs run: {len(history_phase2.history['accuracy'])}")
print(f"  - Time: {phase2_time/60:.1f} minutes")
print(f"  - Final Val Acc: {history_phase2.history['val_accuracy'][-1]:.2%}")
print(f"\n🎯 Best Validation Accuracy: {max(max(history_phase1.history['val_accuracy']), max(history_phase2.history['val_accuracy'])):.2%}")
print("="*80)


In [ ]:


print("Defining prediction and display functions...")

from tensorflow.keras.applications.efficientnet import preprocess_input # Added for correct preprocessing

def predict_food_and_calories(image_path):
    """
    Predict food type and estimate calories from an image.

    Args:
        image_path (str): Path to the food image.

    Returns:
        dict: A dictionary containing prediction results, including food name,
              confidence, calorie information, and top N predictions.
    """
    # Load and preprocess image
    img = image.load_img(image_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    # Make prediction
    predictions = model.predict(img_array, verbose=0)
    predicted_idx = np.argmax(predictions[0])
    predicted_food = class_names[predicted_idx]
    confidence = float(predictions[0][predicted_idx]) * 100

    top_3_idx = np.argsort(predictions[0])[::-1][:3] # Sort in descending order
    top_3_predictions = [
        {"food": class_names[idx], "confidence": float(predictions[0][idx]) * 100}
        for idx in top_3_idx
    ]

    calorie_info = calorie_database.get(predicted_food, {})

    result = {
        "predicted_food": predicted_food,
        "confidence": f"{confidence:.2f}%",
        "calories_per_serving": calorie_info.get("calories_per_serving", "N/A"),
        "serving_size_g": calorie_info.get("serving_size_g", "N/A"),
        "usda_food_name": calorie_info.get("usda_food_name", "N/A"),
        "top_3_predictions": top_3_predictions
    }

    return result


def display_prediction_result(result):
    """
    Prints the prediction results in a user-friendly format.
    """
    print("\n" + "="*70)
    print("🍽️  FOOD PREDICTION RESULT")
    print("="*70)
    print(f"Predicted Food: {result['predicted_food'].replace('_', ' ').title()}")
    print(f"Confidence: {result['confidence']}")
    print(f"\nCalorie Information:")
    print(f"  USDA Matched Name: {result['usda_food_name']}")
    print(f"  Calories per serving: {result['calories_per_serving']} kcal")
    print(f"  Serving size: {result['serving_size_g']}g")
    print(f"\nTop 3 Predictions:")
    for i, pred in enumerate(result['top_3_predictions'], 1):
        food_name = pred['food'].replace('_', ' ').title()
        print(f"  {i}. {food_name}: {pred['confidence']:.2f}%")
    print("="*70 + "\n")


print("✓ Prediction functions created successfully!")

Defining prediction and display functions...
✓ Prediction functions created successfully!


In [ ]:


print("Testing predictions on 5 random images from the training set...\n")


for i in range(5):
    random_idx = np.random.randint(len(train_images))
    test_img_path = train_images[random_idx]

    print(f"\n--- Test {i+1}/5 ---")
    print(f"Processing image: {test_img_path}")

    result = predict_food_and_calories(test_img_path)
    display_prediction_result(result)

    img = image.load_img(test_img_path)
    plt.figure(figsize=(6, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {result['predicted_food'].replace('_', ' ').title()}")
    plt.tight_layout()
    plt.show()

print("\n✓ Sample testing complete!")

In [ ]:


from google.colab import files

print("Use the button below to upload an image of food to test the model!")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nTesting on uploaded image: {filename}")
    result = predict_food_and_calories(filename)
    display_prediction_result(result)

    img = image.load_img(filename)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Predicted: {result['predicted_food'].replace('_', ' ').title()}\nConfidence: {result['confidence']}")
    plt.tight_layout()
    plt.show()